In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
# Import dataset and metadata
X = pd.read_excel("/Users/esther/PycharmProjects/ad_proteomic_clock/training_data/JPST003472.xlsx").set_index("sample_id")
meta = pd.read_excel("/Users/esther/PycharmProjects/ad_proteomic_clock/training_data/JPST003472_meta.xlsx").set_index("sample_id")

X_full = X.copy()
y_full = meta.loc[X_full.index, "age"].astype(float).copy()

# Copy with excluded flagged samples
exclude_samples = ["8M_2", "14M_9"]

X_exclude = X.drop(index=exclude_samples)
y_exclude = meta.loc[X_exclude.index, "age"].astype(float).copy()

print(f"Full: n={X_full.shape[0]}, Reduced: n={X_exclude.shape[0]}")

Full: n=45, Reduced: n=43


In [7]:
def nested_cv(X, y, n_splits=5, n_repeats=20, seed=27):
    outer_cv = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)

    parameters = {
        'model__alpha': np.logspace(-2, 1, 20),
        'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]
    }

    fold_results = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet(max_iter=50000, tol=1e-3)) # increased max iteration and tolerance as model did not converge (>4000 features)
        ])

        inner_cv = RepeatedKFold(n_splits=n_splits, n_repeats=1, random_state=seed)
        search = GridSearchCV(pipe, parameters, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)
        search.fit(X_train, y_train)

        y_pred = search.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)

        fold_results.append({
            'fold': fold_idx,
            'mae': mae,
            'best_alpha': search.best_params_['model__alpha'],
            'best_l1_ratio': search.best_params_['model__l1_ratio'],
            'n_test': len(test_idx)
        })

    return pd.DataFrame(fold_results)

results_full = nested_cv(X_full, y_full)
results_exclude = nested_cv(X_exclude, y_exclude)

print("\n n = 45 ")
print(f"Mean mean absolute error: {results_full['mae'].mean():.3f}± {results_full['mae'].std():.3f}")

print("\n--- n=43 (excl. 8M_2, 14M_9) ---")
print(f"Mean MAE: {results_exclude['mae'].mean():.3f} ± {results_exclude['mae'].std():.3f}")

PicklingError: Could not pickle the task to send it to the workers.